In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import pandas as pd
from housepriceprediction.config import DATA_RAW
from housepriceprediction.transformers import (
    ColumnDropper,
    FeatureEngineer,
    NeighborhoodMedianImputer,
    ConditionalImputer,
)
from housepriceprediction.features import ORDINAL_CATEGORIES

# 1. Load raw data and separate ID / Target
train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
X = train_df.drop(columns=["Id", "SalePrice"])

# 2. Sequentially apply steps 1-4 (implemented transformers)
X_trans = ColumnDropper().fit_transform(X)
X_trans = FeatureEngineer().fit_transform(X_trans)
X_trans = NeighborhoodMedianImputer().fit_transform(X_trans)
X_trans = ConditionalImputer().fit_transform(X_trans)

# 3. Separate into feature groups
ordinal_cols = list(ORDINAL_CATEGORIES.keys())

# Numeric: any column with numeric dtype after transformations (includes 'HasPool')
numeric_cols = X_trans.select_dtypes(include=["number", "int64", "float64"]).columns.tolist()

# Nominal: text/object columns that are NOT ordinal (includes 'MSSubClass' and 'MoSold')
text_cols = X_trans.select_dtypes(include=["object", "string"]).columns.tolist()
nominal_cols = [c for c in text_cols if c not in ordinal_cols]

print("NUMERIC_COLUMNS =", numeric_cols)
print("NOMINAL_COLUMNS =", nominal_cols)

# 4. Integrity check (Validate total coverage and uniqueness)
all_processed = set(X_trans.columns)
all_grouped = set(ordinal_cols) | set(numeric_cols) | set(nominal_cols)

print("\n--- Integrity Check ---")
print("Are all columns covered?", all_processed == all_grouped)
print("Missing columns:", all_processed - all_grouped)
print("Extra/duplicated columns:", all_grouped - all_processed)

# Intersections between groups (must be empty sets)
print("Intersection Numeric & Ordinal:", set(numeric_cols) & set(ordinal_cols))
print("Intersection Nominal & Ordinal:", set(nominal_cols) & set(ordinal_cols))
print("Intersection Numeric & Nominal:", set(numeric_cols) & set(nominal_cols))


NUMERIC_COLUMNS = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal', 'YrSold', 'HasPool']
NOMINAL_COLUMNS = ['MSSubClass', 'MSZoning', 'Alley', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'CentralAir', 'Electrical', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 'MoSold', 'SaleType', 'SaleCondition']

--- Integrity Check ---
Are all columns covered? True
Missing columns: set()
Extra/duplicated columns: set()
Intersection Numeric & Ordinal: set()
Intersection Nominal

In [8]:
import numpy as np
import pandas as pd
from housepriceprediction.config import DATA_RAW

# Load data and target (keeping consistent NA handling)
df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
y_log = df["SalePrice"].apply(np.log1p)

# 1. Calculate candidates
df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]

# New candidates: Baths and Porches
df["TotalBath"] = df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
df["TotalPorchSF"] = df["WoodDeckSF"] + df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] + df["ScreenPorch"]

# 2. Compare correlations with log1p(SalePrice)
sf_cols = ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "GrLivArea", "TotalSF"]
age_cols = ["YearBuilt", "HouseAge", "YearRemodAdd", "RemodelAge"]
bath_cols = ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath", "TotalBath"]
porch_cols = ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "TotalPorchSF"]

print("--- Surface ---")
print(df[sf_cols].corrwith(y_log).sort_values(ascending=False))

print("\n--- Baths ---")
print(df[bath_cols].corrwith(y_log).sort_values(ascending=False))

print("\n--- Porch/Outside ---")
print(df[porch_cols].corrwith(y_log).sort_values(ascending=False))

print("\n--- Age ---")
# Using abs() for age because lower age = higher price (negative correlation)
print(df[age_cols].corrwith(y_log).abs().sort_values(ascending=False))

# 3. Search for strange values (time travel)
invalid_age = df[df["HouseAge"] < 0]
invalid_remodel = df[df["RemodelAge"] < 0]

print(f"\nHouses sold before being built: {len(invalid_age)}")
if not invalid_age.empty:
    print(invalid_age[["Id", "YearBuilt", "YrSold", "HouseAge"]])

print(f"\nHouses sold before being renovated: {len(invalid_remodel)}")
if not invalid_remodel.empty:
    print(invalid_remodel[["Id", "YearRemodAdd", "YrSold", "RemodelAge"]])

--- Surface ---
TotalSF        0.777297
GrLivArea      0.700927
TotalBsmtSF    0.612134
1stFlrSF       0.596981
2ndFlrSF       0.319300
dtype: float64

--- Baths ---
TotalBath       0.673011
FullBath        0.594771
HalfBath        0.313982
BsmtFullBath    0.236224
BsmtHalfBath   -0.005149
dtype: float64

--- Porch/Outside ---
TotalPorchSF     0.398387
WoodDeckSF       0.334135
OpenPorchSF      0.321053
ScreenPorch      0.121208
3SsnPorch        0.054900
EnclosedPorch   -0.149050
dtype: float64

--- Age ---
HouseAge        0.587290
YearBuilt       0.586570
RemodelAge      0.568136
YearRemodAdd    0.565608
dtype: float64

Houses sold before being built: 0

Houses sold before being renovated: 1
      Id  YearRemodAdd  YrSold  RemodelAge
523  524          2008    2007          -1
